# Лабораторная работа 4

## Инкин Артем Игоревич, 6401-010302D

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
device = '/gpu:0'

In [ ]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [ ]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(channel_1, 5, padding='same', activation='relu')
        self.conv2 = tf.keras.layers.Conv2D(channel_2, 3, padding='same', activation='relu')
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [ ]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [6]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [ ]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.237762451171875, Accuracy: 10.9375, Val Loss: 2.961097240447998, Val Accuracy: 14.90000057220459
Iteration 100, Epoch 1, Loss: 2.200409173965454, Accuracy: 29.099628448486328, Val Loss: 1.84576416015625, Val Accuracy: 38.900001525878906
Iteration 200, Epoch 1, Loss: 2.0586659908294678, Accuracy: 32.57151794433594, Val Loss: 1.796156644821167, Val Accuracy: 40.20000076293945
Iteration 300, Epoch 1, Loss: 1.9863234758377075, Accuracy: 34.28675079345703, Val Loss: 1.8620145320892334, Val Accuracy: 39.20000076293945
Iteration 400, Epoch 1, Loss: 1.9234079122543335, Accuracy: 36.01543045043945, Val Loss: 1.7090611457824707, Val Accuracy: 42.39999771118164
Iteration 500, Epoch 1, Loss: 1.879597783088684, Accuracy: 37.072731018066406, Val Loss: 1.6337039470672607, Val Accuracy: 45.20000076293945
Iteration 600, Epoch 1, Loss: 1.8515105247497559, Accuracy: 37.96017074584961, Val Loss: 1.6791608333587646, Val Accuracy: 41.900001525878906
Iteration 700, Epoch 1, Loss

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [ ]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.361072301864624, Accuracy: 9.375, Val Loss: 2.344715118408203, Val Accuracy: 9.600000381469727
Iteration 100, Epoch 1, Loss: 1.8966450691223145, Accuracy: 32.22462844848633, Val Loss: 1.6272510290145874, Val Accuracy: 44.10000228881836
Iteration 200, Epoch 1, Loss: 1.7407056093215942, Accuracy: 38.557212829589844, Val Loss: 1.4680942296981812, Val Accuracy: 49.5
Iteration 300, Epoch 1, Loss: 1.6584830284118652, Accuracy: 41.39846420288086, Val Loss: 1.423302412033081, Val Accuracy: 50.70000457763672
Iteration 400, Epoch 1, Loss: 1.5916110277175903, Accuracy: 43.82793045043945, Val Loss: 1.368804931640625, Val Accuracy: 50.900001525878906
Iteration 500, Epoch 1, Loss: 1.5470294952392578, Accuracy: 45.34368896484375, Val Loss: 1.318602204322815, Val Accuracy: 55.80000305175781
Iteration 600, Epoch 1, Loss: 1.5188775062561035, Accuracy: 46.401832580566406, Val Loss: 1.300737738609314, Val Accuracy: 54.5
Iteration 700, Epoch 1, Loss: 1.4931038618087769, Accura

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [ ]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.020972728729248, Accuracy: 9.375, Val Loss: 2.9502370357513428, Val Accuracy: 11.800000190734863
Iteration 100, Epoch 1, Loss: 2.2242958545684814, Accuracy: 28.496286392211914, Val Loss: 1.9035948514938354, Val Accuracy: 38.10000228881836
Iteration 200, Epoch 1, Loss: 2.067897319793701, Accuracy: 32.361629486083984, Val Loss: 1.897290825843811, Val Accuracy: 39.0
Iteration 300, Epoch 1, Loss: 1.991902232170105, Accuracy: 34.323089599609375, Val Loss: 1.894311785697937, Val Accuracy: 35.29999923706055
Iteration 400, Epoch 1, Loss: 1.9293895959854126, Accuracy: 36.01153564453125, Val Loss: 1.7180267572402954, Val Accuracy: 41.900001525878906
Iteration 500, Epoch 1, Loss: 1.8862500190734863, Accuracy: 37.0789680480957, Val Loss: 1.6718368530273438, Val Accuracy: 43.099998474121094
Iteration 600, Epoch 1, Loss: 1.8558666706085205, Accuracy: 37.949771881103516, Val Loss: 1.700921893119812, Val Accuracy: 42.29999923706055
Iteration 700, Epoch 1, Loss: 1.82899951

Альтернативный менее гибкий способ обучения:

In [ ]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 80s 104ms/step - loss: 1.8278 - sparse_categorical_accuracy: 0.3876 - val_loss: 1.6119 - val_sparse_categorical_accuracy: 0.4590
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - loss: 1.5903 - sparse_categorical_accuracy: 0.4480


[1.590291142463684, 0.4480000138282776]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [ ]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 5, padding='same', activation='relu', input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.3298559188842773, Accuracy: 12.5, Val Loss: 2.316945791244507, Val Accuracy: 9.5
Iteration 100, Epoch 1, Loss: 2.1199560165405273, Accuracy: 23.236387252807617, Val Loss: 1.9632643461227417, Val Accuracy: 32.10000228881836
Iteration 200, Epoch 1, Loss: 2.013676643371582, Accuracy: 27.92288589477539, Val Loss: 1.8364375829696655, Val Accuracy: 37.20000076293945
Iteration 300, Epoch 1, Loss: 1.9482975006103516, Accuracy: 30.876245498657227, Val Loss: 1.7663893699645996, Val Accuracy: 41.20000076293945
Iteration 400, Epoch 1, Loss: 1.8857429027557373, Accuracy: 33.358009338378906, Val Loss: 1.6888573169708252, Val Accuracy: 44.60000228881836
Iteration 500, Epoch 1, Loss: 1.8370306491851807, Accuracy: 35.042415618896484, Val Loss: 1.6143699884414673, Val Accuracy: 44.70000076293945
Iteration 600, Epoch 1, Loss: 1.8012574911117554, Accuracy: 36.3118782043457, Val Loss: 1.570473551750183, Val Accuracy: 46.79999923706055
Iteration 700, Epoch 1, Loss: 1.7676749229

In [ ]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 100s 130ms/step - loss: 1.6218 - sparse_categorical_accuracy: 0.4252 - val_loss: 1.3403 - val_sparse_categorical_accuracy: 0.5380
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - loss: 1.3786 - sparse_categorical_accuracy: 0.5128


[1.3785871267318726, 0.5127999782562256]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [ ]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [ ]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.7075576782226562, Accuracy: 14.0625, Val Loss: 2.9618263244628906, Val Accuracy: 12.800000190734863
Iteration 100, Epoch 1, Loss: 2.2218477725982666, Accuracy: 28.96039581298828, Val Loss: 1.8724218606948853, Val Accuracy: 37.400001525878906
Iteration 200, Epoch 1, Loss: 2.0716371536254883, Accuracy: 32.86691665649414, Val Loss: 1.820035457611084, Val Accuracy: 38.0
Iteration 300, Epoch 1, Loss: 1.9963070154190063, Accuracy: 34.727989196777344, Val Loss: 1.863792896270752, Val Accuracy: 37.0
Iteration 400, Epoch 1, Loss: 1.9328899383544922, Accuracy: 36.30377197265625, Val Loss: 1.7356518507003784, Val Accuracy: 42.20000076293945
Iteration 500, Epoch 1, Loss: 1.8893457651138306, Accuracy: 37.337825775146484, Val Loss: 1.6476364135742188, Val Accuracy: 42.099998474121094
Iteration 600, Epoch 1, Loss: 1.8597489595413208, Accuracy: 38.11355972290039, Val Loss: 1.6783978939056396, Val Accuracy: 43.20000076293945
Iteration 700, Epoch 1, Loss: 1.835416316986084,

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

**Эксперимент 1 - Baseline**
- Архитектура: 2 conv слоя (32, 64 фильтра) => MaxPool => FC
- Оптимизатор: SGD, lr=0.01, без momentum
- Регуляризация: нет
- Результат: val accuracy 65.3%

Базовая модель без регуляризации быстро переобучается, сходится медленно из-за отсутствия momentum.

In [7]:
class ConvNet_Exp1(tf.keras.Model):
    def __init__(self):
        super(ConvNet_Exp1, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(10, activation='softmax')

    def call(self, x, training=False):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

print_every = 700
num_epochs = 10

def model_init_fn():
    return ConvNet_Exp1()

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=1e-2)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 2.3264098167419434, Accuracy: 14.0625, Val Loss: 2.339350938796997, Val Accuracy: 9.800000190734863
Iteration 700, Epoch 1, Loss: 1.7106269598007202, Accuracy: 39.829261779785156, Val Loss: 1.4830416440963745, Val Accuracy: 48.69999694824219
Iteration 1400, Epoch 2, Loss: 1.3454406261444092, Accuracy: 52.6550178527832, Val Loss: 1.2834675312042236, Val Accuracy: 55.0
Iteration 2100, Epoch 3, Loss: 1.1944376230239868, Accuracy: 58.290313720703125, Val Loss: 1.231821060180664, Val Accuracy: 57.5
Iteration 2800, Epoch 4, Loss: 1.0912479162216187, Accuracy: 62.22664260864258, Val Loss: 1.1632064580917358, Val Accuracy: 58.79999923706055
Iteration 3500, Epoch 5, Loss: 1.0112228393554688, Accuracy: 64.93850708007812, Val Loss: 1.1295315027236938, Val Accuracy: 61.599998474121094
Iteration 4200, Epoch 6, Loss: 0.9492560029029846, Accuracy: 67.47810363769531, Val Loss: 1.0492585897445679, Val Accuracy: 64.60000610351562
Iteration 4900, Epoch 7, Loss: 0.8951889276504

**Эксперимент 2 - BatchNorm + Dropout + SGD Nesterov**
- Архитектура: 2 conv слоя + BatchNorm + MaxPool => FC(256) + Dropout(0.5) => FC
- Оптимизатор: SGD, lr=0.01, momentum=0.9, nesterov=True
- Регуляризация: BatchNorm + Dropout 0.5
- Результат: val accuracy 68.3%

Добавление BatchNorm ускорило сходимость, Dropout снизил переобучение. Nesterov momentum дал заметный прирост по сравнению с baseline.

In [8]:
class ConvNet_Exp2(tf.keras.Model):
    def __init__(self):
        super(ConvNet_Exp2, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu')
        self.drop = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

    def call(self, x, training=False):
        x = tf.keras.activations.relu(self.bn1(self.conv1(x), training=training))
        x = tf.keras.activations.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop(x, training=training)
        x = self.fc2(x)
        return x

print_every = 700
num_epochs = 10

def model_init_fn():
    return ConvNet_Exp2()

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9, nesterov=True)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.7919328212738037, Accuracy: 6.25, Val Loss: 2.6746416091918945, Val Accuracy: 11.200000762939453
Iteration 700, Epoch 1, Loss: 2.0990397930145264, Accuracy: 20.64015769958496, Val Loss: 1.748266339302063, Val Accuracy: 36.599998474121094
Iteration 1400, Epoch 2, Loss: 1.924290418624878, Accuracy: 23.905019760131836, Val Loss: 1.6562492847442627, Val Accuracy: 41.5
Iteration 2100, Epoch 3, Loss: 1.8604360818862915, Accuracy: 25.99681282043457, Val Loss: 1.6131829023361206, Val Accuracy: 42.79999923706055
Iteration 2800, Epoch 4, Loss: 1.757110834121704, Accuracy: 30.498260498046875, Val Loss: 1.4840211868286133, Val Accuracy: 46.099998474121094
Iteration 3500, Epoch 5, Loss: 1.5901820659637451, Accuracy: 38.729976654052734, Val Loss: 1.3025789260864258, Val Accuracy: 56.900001525878906
Iteration 4200, Epoch 6, Loss: 1.4241464138031006, Accuracy: 46.51701354980469, Val Loss: 1.1986196041107178, Val Accuracy: 58.60000228881836
Iteration 4900, Epoch 7, Loss: 1

**Эксперимент 3 - Adam вместо SGD**
- Архитектура: та же что в эксперименте 2
- Оптимизатор: Adam, lr=0.001
- Регуляризация: BatchNorm + Dropout 0.5
- Результат: val accuracy 49.3%

Неожиданно слабый результат 49.3% против 68.3% у SGD с той же архитектурой. Модель медленно выходила из плато (train accuracy держалась около 26% несколько эпох подряд). 
Adam не всегда превосходит SGD: при взаимодействии с BatchNorm и малом числе фильтров адаптивный lr может замедлить сходимость. SGD с Nesterov оказался устойчивее.

In [9]:
class ConvNet_Exp3(tf.keras.Model):
    def __init__(self):
        super(ConvNet_Exp3, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu')
        self.drop = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

    def call(self, x, training=False):
        x = tf.keras.activations.relu(self.bn1(self.conv1(x), training=training))
        x = tf.keras.activations.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop(x, training=training)
        x = self.fc2(x)
        return x

print_every = 700
num_epochs = 10

def model_init_fn():
    return ConvNet_Exp3()

def optimizer_init_fn():
    return tf.keras.optimizers.Adam(learning_rate=1e-3)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.678969621658325, Accuracy: 3.125, Val Loss: 3.8824965953826904, Val Accuracy: 14.0
Iteration 700, Epoch 1, Loss: 2.1567227840423584, Accuracy: 19.43874740600586, Val Loss: 1.7580193281173706, Val Accuracy: 27.799999237060547
Iteration 1400, Epoch 2, Loss: 1.9674110412597656, Accuracy: 20.888286590576172, Val Loss: 1.72005033493042, Val Accuracy: 35.60000228881836
Iteration 2100, Epoch 3, Loss: 1.9324538707733154, Accuracy: 22.105669021606445, Val Loss: 1.7078365087509155, Val Accuracy: 38.0
Iteration 2800, Epoch 4, Loss: 1.9110740423202515, Accuracy: 23.527584075927734, Val Loss: 1.6930761337280273, Val Accuracy: 36.5
Iteration 3500, Epoch 5, Loss: 1.8806754350662231, Accuracy: 24.649599075317383, Val Loss: 1.646631121635437, Val Accuracy: 36.599998474121094
Iteration 4200, Epoch 6, Loss: 1.8773717880249023, Accuracy: 25.265329360961914, Val Loss: 1.5949780941009521, Val Accuracy: 43.900001525878906
Iteration 4900, Epoch 7, Loss: 1.8602114915847778, Accura

**Эксперимент 4 - Финальная модель**
- Архитектура: 4 conv слоя (32=>64=>128=>128) + 2x MaxPool => FC(512) + Dropout => FC
- Оптимизатор: Adam, lr=0.001
- Регуляризация: BatchNorm после каждого conv + Dropout 0.5
- Результат: val accuracy 78.3%

Более глубокая архитектура с двумя блоками позволила извлечь более сложные признаки. BatchNorm стабилизировал обучение, два MaxPooling снизили размерность и количество параметров.

In [10]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(2)
        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same')
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(128, 3, padding='same')
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(2)
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(512, activation='relu')
        self.drop = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = tf.keras.activations.relu(self.bn1(self.conv1(input_tensor), training=training))
        x = tf.keras.activations.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool1(x)
        x = tf.keras.activations.relu(self.bn3(self.conv3(x), training=training))
        x = tf.keras.activations.relu(self.bn4(self.conv4(x), training=training))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.7063746452331543, Accuracy: 9.375, Val Loss: 2.7470128536224365, Val Accuracy: 10.699999809265137
Iteration 700, Epoch 1, Loss: 1.8630189895629883, Accuracy: 34.91217803955078, Val Loss: 1.3802735805511475, Val Accuracy: 50.400001525878906
Iteration 1400, Epoch 2, Loss: 1.366675615310669, Accuracy: 49.52017593383789, Val Loss: 1.1513746976852417, Val Accuracy: 58.89999771118164
Iteration 2100, Epoch 3, Loss: 1.1915452480316162, Accuracy: 56.73330307006836, Val Loss: 1.0724667310714722, Val Accuracy: 62.400001525878906
Iteration 2800, Epoch 4, Loss: 1.0756772756576538, Accuracy: 61.12698745727539, Val Loss: 0.8588931560516357, Val Accuracy: 70.5
Iteration 3500, Epoch 5, Loss: 0.9898260831832886, Accuracy: 64.39501953125, Val Loss: 0.9432445764541626, Val Accuracy: 65.30000305175781
Iteration 4200, Epoch 6, Loss: 0.9185645580291748, Accuracy: 67.01061248779297, Val Loss: 0.8034958243370056, Val Accuracy: 70.80000305175781
Iteration 4900, Epoch 7, Loss: 0.867


### Результаты по экспериментам:

**Эксперимент 1 - Baseline (65.3%)**
Простая архитектура из 2 свёрточных слоёв без какой-либо регуляризации, SGD без momentum. Модель стабильно обучается, но медленно сходится - точность растёт равномерно все 10 эпох и к концу достигает 65.3%. Видно что train accuracy (73.6%) заметно выше val accuracy, что говорит о переобучении.

**Эксперимент 2 - BN + Dropout + SGD Nesterov (68.3%)**
Та же архитектура, но добавлены BatchNorm после каждого свёрточного слоя и Dropout 0.5 перед выходным слоем. SGD заменён на SGD с Nesterov momentum. Модель обучалась нестабильно в первых эпохах (val accuracy 11.2% на старте), но к концу вышла на 68.3%. BatchNorm стабилизировал обучение, Dropout снизил разрыв между train и val accuracy по сравнению с baseline.

**Эксперимент 3 - Adam (49.3%)**
Архитектура из эксперимента 2, оптимизатор заменён на Adam. Неожиданно слабый результат - 49.3% против 68.3% у SGD с той же архитектурой. Причина в том, что Adam с lr=0.001 плохо взаимодействует с BatchNorm при данной архитектуре - модель медленно выходила из плато (train accuracy 26% на протяжении нескольких эпох). Adam не всегда превосходит SGD, особенно на свёрточных сетях с нормализацией.

**Эксперимент 4 - Финальная модель (78.3%)**
Более глубокая архитектура = 4 свёрточных слоя в двух блоках с MaxPooling между ними, FC(512) с Dropout, Adam. Лучший результат - 78.3% val accuracy при 75.5% train accuracy. Примечательно что val accuracy даже выше train, что говорит об эффективной регуляризации. Два блока свёрточных слоёв позволили модели извлекать признаки разного уровня абстракции.

### Общие выводы:
- Наибольший прирост качества дало увеличение глубины архитектуры (+13% к baseline)
- BatchNorm обязателен при использовании Dropout - без него сходимость нестабильна
- Adam не всегда лучше SGD: при неудачном lr он может застрять в плато
- Два блока MaxPooling снизили размерность и улучшили обобщающую способность модели
- Финальная модель превысила целевой порог в 70% и достигла 78.3% val accuracy